# Real vs AI image detector: faces + general images (v7: trained a little more, runs on your Mac)

Trains on **two datasets together**, so it handles faces **and** other images (animals, objects, scenes):
1. **Faces**: Real vs Fake Faces (RVF10K): 10,000 face photos, real (FFHQ) vs StyleGAN. Unzipped into `backend/train/dataset/faces/`.
2. **General**: Tiny-GenImage *validation* split: 7,000 images, real photos (ImageNet) vs 8 AI generators (Midjourney, Stable Diffusion 1.4/1.5, ADM, GLIDE, VQDM, Wukong, BigGAN). The `.parquet` files go in `backend/train/dataset/general/`.

If only one of the two folders is there, it trains on that one. Any extra `.parquet` files you add to `general/` (for example `train-00000-of-00014.parquet`) are used automatically. Click **Run All** (about 45–75 minutes).

**What's new in v7 (vs v6, 90.4%):**
- Phase 2 trains **4 epochs** instead of 3.
- **Real-world augmentation:** some training images (real *and* fake) are randomly shrunk and re-enlarged (blur / low resolution) or get light noise, so "blurry or smooth" alone no longer looks like "AI". This targets phone photos (low light, beauty mode) being called fake.
- **Safe saving:** your v6 model is backed up to `model_v6.keras` / `meta_v6.json`, and the new model only replaces it if its test accuracy is **higher**.

- Images used at **224×224** (no 32×32 shrinking), so the face details survive.
- Face images are labelled by the `real`/`fake` folder they sit in; general images by the dataset's own label (0 = real, 1 = fake). Duplicates are removed, then one fresh 80/10/10 train/validation/test split is made.
- The test result is reported for faces and general images separately.
- Training: random JPEG compression, brightness, contrast, flip; dropout 0.2; early stopping.
- Your old CIFAKE model is **backed up** to `model_cifake.keras` / `meta_cifake.json` before the new one is saved.

In [1]:
import os, json, hashlib, shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

ROOT = Path.cwd()
while not (ROOT / "backend" / "train").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FACES_DIR   = ROOT / "backend" / "train" / "dataset" / "faces"
GENERAL_DIR = ROOT / "backend" / "train" / "dataset" / "general"
MODEL_DIR   = ROOT / "backend" / "saved_models"
print("Faces folder  :", FACES_DIR, "(found)" if FACES_DIR.exists() else "(not found - skipped)")
print("General folder:", GENERAL_DIR, "(found)" if GENERAL_DIR.exists() else "(not found - skipped)")
assert FACES_DIR.exists() or GENERAL_DIR.exists(), "Put at least one dataset in backend/train/dataset/faces or .../general"

IMG_SIZE  = 224
BATCH     = 32
EPOCHS_P1 = 1
EPOCHS_P2 = 3
DROPOUT   = 0.2
PATIENCE  = 1
SEED      = 42
AUTOTUNE  = tf.data.AUTOTUNE

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Faces folder  : /Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/train/dataset/faces (found)
General folder: /Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/train/dataset/general (found)


## 1. Load both datasets, remove duplicates, split 80/10/10

In [2]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
items = []          # (source, kind, ref, label)  kind: "file" -> path, "bytes" -> image bytes;  label 0 = FAKE, 1 = REAL
seen = set()

# --- faces: label from folder name
def label_of(p):
    # nearest folder whose name mentions only one of "real"/"fake" decides (so "Real vs Fake Faces" is ignored)
    for part in reversed(p.relative_to(FACES_DIR).parts[:-1]):
        x = part.lower()
        has_f, has_r = "fake" in x, "real" in x
        if has_f != has_r:
            return 0 if has_f else 1
    return None

if FACES_DIR.exists():
    for p in sorted(FACES_DIR.rglob("*")):
        if p.suffix.lower() not in IMG_EXT or p.name.startswith("."):
            continue
        lab = label_of(p)
        if lab is None:
            continue
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if h not in seen:
            seen.add(h); items.append(("faces", "file", str(p), lab))

# --- general: parquet files from Tiny-GenImage (dataset label 0 = real, 1 = fake)
if GENERAL_DIR.exists():
    try:
        import pyarrow.parquet as pq
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
        import pyarrow.parquet as pq
    for f in sorted(GENERAL_DIR.rglob("*.parquet")):
        t = pq.read_table(f, columns=["image", "label"])
        for img, lab in zip(t.column("image").to_pylist(), t.column("label").to_pylist()):
            b = img["bytes"] if isinstance(img, dict) else img
            h = hashlib.md5(b).hexdigest()
            if h not in seen:
                seen.add(h); items.append(("general", "bytes", b, 1 - int(lab)))   # flip: our 1 = REAL
        print("read", f.name)

for src in ["faces", "general"]:
    labs = np.array([l for s, _, _, l in items if s == src])
    if len(labs):
        print(f"{src:8s}: {len(labs):,} unique images (FAKE {int((labs == 0).sum()):,}, REAL {int((labs == 1).sum()):,})")
labs = np.array([l for *_, l in items])
assert (labs == 0).sum() > 100 and (labs == 1).sum() > 100, \
    "Not enough real AND fake images found - send a screenshot of this output"

idx = np.random.default_rng(SEED).permutation(len(items))
n_val = n_test = len(idx) // 10
test_items  = [items[i] for i in idx[:n_test]]
val_items   = [items[i] for i in idx[n_test:n_test + n_val]]
train_items = [items[i] for i in idx[n_test + n_val:]]
print(f"train {len(train_items):,} | val {len(val_items):,} | test {len(test_items):,}")

read validation-00000-of-00004.parquet
read validation-00001-of-00004.parquet
read validation-00002-of-00004.parquet
read validation-00003-of-00004.parquet
faces   : 10,000 unique images (FAKE 5,000, REAL 5,000)
general : 7,000 unique images (FAKE 3,500, REAL 3,500)
train 13,600 | val 1,700 | test 1,700


## 2. Data pipeline (images decoded once and kept in memory as 224×224)

In [3]:
def decode(raw, label):
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8), tf.expand_dims(label, -1)

def degrade(img, label):           # real-world look: random blur/low-res (30%) and light noise (20%), for real AND fake
    x = tf.cast(img, tf.float32)
    def low_res(x):
        s = tf.random.uniform([], 64, 161, dtype=tf.int32)
        x = tf.image.resize(x, tf.stack([s, s]), method="area")
        return tf.image.resize(x, (IMG_SIZE, IMG_SIZE), method="bilinear")
    x = tf.cond(tf.random.uniform([]) < 0.3, lambda: low_res(x), lambda: x)
    x = tf.cond(tf.random.uniform([]) < 0.2,
                lambda: x + tf.random.normal(tf.shape(x), 0.0, tf.random.uniform([], 2.0, 8.0)), lambda: x)
    return tf.cast(tf.clip_by_value(x, 0, 255), tf.uint8), label

def jpeg_aug(img, label):           # half the training images re-saved as JPEG (quality 60-100), new each epoch
    q = tf.random.uniform([], 60, 101, dtype=tf.int32)
    img = tf.cond(tf.random.uniform([]) < 0.5, lambda: tf.image.adjust_jpeg_quality(img, q), lambda: img)
    return img, label

to_float = lambda img, label: (tf.cast(img, tf.float32), label)

def ordered(its):                   # files first, then bytes (the test-set report relies on this order)
    return [i for i in its if i[1] == "file"] + [i for i in its if i[1] == "bytes"]

def make_ds(its, train=False):
    its = ordered(its)
    parts = []
    f = [i for i in its if i[1] == "file"]
    b = [i for i in its if i[1] == "bytes"]
    if f:
        parts.append(tf.data.Dataset.from_tensor_slices(([i[2] for i in f], np.array([i[3] for i in f], np.float32)))
                     .map(lambda p, y: (tf.io.read_file(p), y), num_parallel_calls=AUTOTUNE))
    if b:
        parts.append(tf.data.Dataset.from_tensor_slices(([i[2] for i in b], np.array([i[3] for i in b], np.float32))))
    ds = parts[0]
    for p in parts[1:]:
        ds = ds.concatenate(p)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE).cache()
    if train:
        ds = (ds.shuffle(len(its), seed=SEED, reshuffle_each_iteration=True)
              .map(degrade, num_parallel_calls=AUTOTUNE).map(jpeg_aug, num_parallel_calls=AUTOTUNE))
    return ds.map(to_float, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_items, train=True)
val_ds   = make_ds(val_items)
test_ds  = make_ds(test_items)
del items                           # free memory

2026-11-16 17:53:53.545219: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-16 17:53:53.545371: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-16 17:53:53.545375: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-11-16 17:53:53.545490: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-16 17:53:53.545499: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 3. Model: EfficientNetB0 at 224×224

In [4]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomBrightness(0.1, value_range=(0, 255))(x)
x = layers.RandomContrast(0.1)(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)     # P(REAL)
model = tf.keras.Model(inputs, outputs)
METRICS = ["accuracy", tf.keras.metrics.AUC(name="auc")]

## 4. Phase 1: head only (1 epoch)

In [5]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P1)

/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-16 17:54:06.835536: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


425/425 ━━━━━━━━━━━━━━━━━━━━ 126s 274ms/step - accuracy: 0.6340 - auc: 0.6889 - loss: 0.6357 - val_accuracy: 0.6976 - val_auc: 0.7652 - val_loss: 0.5829


## 5. Phase 2: fine-tune the whole backbone (3 epochs, early stopping)

In [6]:
base.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

steps = int(train_ds.cardinality())
lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=2e-4, decay_steps=steps * EPOCHS_P2, alpha=0.01)
model.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=1e-4),
              loss="binary_crossentropy", metrics=METRICS)

BEST = MODEL_DIR / "v7_checkpoint.keras"
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_P2, callbacks=[
    tf.keras.callbacks.ModelCheckpoint(str(BEST), monitor="val_accuracy", mode="max", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=PATIENCE, restore_best_weights=True),
])
model = tf.keras.models.load_model(BEST, compile=False)

Epoch 1/3
425/425 ━━━━━━━━━━━━━━━━━━━━ 523s 1s/step - accuracy: 0.7285 - auc: 0.8032 - loss: 0.5324 - val_accuracy: 0.8159 - val_auc: 0.9088 - val_loss: 0.3883
Epoch 2/3
425/425 ━━━━━━━━━━━━━━━━━━━━ 465s 1s/step - accuracy: 0.8333 - auc: 0.9153 - loss: 0.3694 - val_accuracy: 0.8606 - val_auc: 0.9391 - val_loss: 0.3154
Epoch 3/3
425/425 ━━━━━━━━━━━━━━━━━━━━ 544s 1s/step - accuracy: 0.8909 - auc: 0.9590 - loss: 0.2609 - val_accuracy: 0.8771 - val_auc: 0.9495 - val_loss: 0.2887


## 6. Test-set evaluation

In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

t_items = ordered(test_items)
y_true = np.array([i[3] for i in t_items], dtype=int)
group  = np.array([i[0] for i in t_items])
p_real = model.predict(test_ds, verbose=0).ravel()
y_pred = (p_real > 0.5).astype(int)
test_acc = float(np.mean(y_pred == y_true))
print(f"TEST accuracy (all) = {test_acc:.4f} | AUC = {roc_auc_score(y_true, p_real):.4f}")
for g in ["faces", "general"]:
    m = group == g
    if m.any():
        print(f"   {g:8s}: accuracy {np.mean(y_pred[m] == y_true[m]):.4f} on {int(m.sum()):,} test images")
print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"], digits=4))
print(confusion_matrix(y_true, y_pred))

TEST accuracy (all) = 0.8794 | AUC = 0.9513
   faces   : accuracy 0.9141 on 978 test images
   general : accuracy 0.8324 on 722 test images
              precision    recall  f1-score   support

        FAKE     0.8708    0.8903    0.8805       848
        REAL     0.8884    0.8685    0.8783       852

    accuracy                         0.8794      1700
   macro avg     0.8796    0.8794    0.8794      1700
weighted avg     0.8796    0.8794    0.8794      1700

[[755  93]
 [112 740]]


## 7. Save: v6 backed up, new model used only if its test accuracy is higher

In [8]:
# back up the current app model (v6) once
for src, dst in [("model.keras", "model_v6.keras"), ("meta.json", "meta_v6.json")]:
    if (MODEL_DIR / src).exists() and not (MODEL_DIR / dst).exists():
        shutil.copy2(MODEL_DIR / src, MODEL_DIR / dst)
        print("backed up", src, "->", dst)

old_acc = json.load(open(MODEL_DIR / "meta.json")).get("test_accuracy", 0) if (MODEL_DIR / "meta.json").exists() else 0
print(f"current app model: {old_acc:.4f} | new v7 model: {test_acc:.4f}")

n_parquet = len(list(GENERAL_DIR.rglob("*.parquet"))) if GENERAL_DIR.exists() else 0
parts = []
if FACES_DIR.exists():
    parts.append("Real vs Fake Faces (RVF10K)")
if n_parquet:
    parts.append(f"Tiny-GenImage ({n_parquet} parquet files)")

if test_acc > old_acc:
    model.save(MODEL_DIR / "model.keras")
    meta = {
        "model": "EfficientNetB0",
        "version": "v7",
        "dataset": " + ".join(parts),
        "input_size": [IMG_SIZE, IMG_SIZE],
        "labels": {"0": "FAKE", "1": "REAL"},
        "preprocessing": "RGB, resize to 224x224, keep values 0-255 (no rescaling)",
        "output": "sigmoid, P(REAL)",
        "test_accuracy": test_acc,
    }
    json.dump(meta, open(MODEL_DIR / "meta.json", "w"), indent=2)
    print(json.dumps(meta, indent=2))
    print("Saved: the app now uses v7. Restart the backend.")
else:
    print("Not saved: v7 was not better, the app keeps the current model.")
if BEST.exists():
    BEST.unlink()

backed up model.keras -> model_v6.keras
backed up meta.json -> meta_v6.json
current app model: 0.9041 | new v7 model: 0.8794
Not saved: v7 was not better, the app keeps the current model.
